# EPA Water Quality: Merge Stations + Measurements

In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
cleaned_path = r'../../../../data/tabular/water-quality/cleaned'

df_wq = pd.read_csv(
    f'{cleaned_path}/epa-wq-clean.csv',
    parse_dates=['ActivityStartDateTime', 'AnalysisStartDate', 'LastUpdated']
)
df_station = pd.read_csv(f'{cleaned_path}/epa-stations-clean.csv')

print(f'Water quality rows: {len(df_wq):,}')
print(f'Stations rows:      {len(df_station):,}')

In [ ]:
# Step 1: Inspect join key overlap
wq_stations = set(df_wq['MonitoringLocationIdentifier'].dropna())
station_ids = set(df_station['MonitoringLocationIdentifier'].dropna())

print(f'Unique stations in WQ data:       {len(wq_stations)}')
print(f'Unique stations in stations file:  {len(station_ids)}')
print(f'Overlap:                           {len(wq_stations & station_ids)}')
print(f'WQ stations NOT in stations file:  {len(wq_stations - station_ids)}')

In [ ]:
# Step 2: Merge water quality measurements with station metadata
# Left join keeps all WQ rows; station metadata added where available
df_merged = df_wq.merge(
    df_station,
    on='MonitoringLocationIdentifier',
    how='left'
)

print(f'Merged shape: {df_merged.shape}')
df_merged.head()

In [ ]:
# Step 3: Resolve latitude/longitude columns
# WQ data has activity-level lat/lon; stations file has fixed station lat/lon
# Use station lat/lon as authoritative; fall back to activity lat/lon
wq_lat = 'ActivityLocation/LatitudeMeasure'
wq_lon = 'ActivityLocation/LongitudeMeasure'

if 'LatitudeMeasure' in df_merged.columns:
    df_merged['latitude'] = df_merged['LatitudeMeasure'].combine_first(
        pd.to_numeric(df_merged.get(wq_lat), errors='coerce')
    )
    df_merged['longitude'] = df_merged['LongitudeMeasure'].combine_first(
        pd.to_numeric(df_merged.get(wq_lon), errors='coerce')
    )
    df_merged.drop(columns=['LatitudeMeasure', 'LongitudeMeasure', wq_lat, wq_lon],
                   errors='ignore', inplace=True)
else:
    df_merged['latitude'] = pd.to_numeric(df_merged.get(wq_lat), errors='coerce')
    df_merged['longitude'] = pd.to_numeric(df_merged.get(wq_lon), errors='coerce')
    df_merged.drop(columns=[wq_lat, wq_lon], errors='ignore', inplace=True)

print(f'Missing latitude:  {df_merged["latitude"].isnull().sum()}')
print(f'Missing longitude: {df_merged["longitude"].isnull().sum()}')

In [ ]:
# Step 4: Final inspection
print(f'Final shape: {df_merged.shape}')
print('\nColumns:', df_merged.columns.tolist())
print('\nMissing values:')
print(df_merged.isnull().sum())

In [ ]:
# Step 5: Save merged data
out_path = '../../../../data/tabular/water-quality/cleaned'
os.makedirs(out_path, exist_ok=True)
df_merged.to_csv(f'{out_path}/epa-merged.csv', index=False)
print(f'Saved {len(df_merged):,} rows → {out_path}/epa-merged.csv')